<a href="https://colab.research.google.com/github/usami0jp/ai2023-old/blob/main/rag_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# 最小構成の RAG（検索拡張生成）サンプル
#   Google Colab でそのまま動きます（GPU推奨、CPUでも可）
#   APIキー不要・すべて無料モデル
#
#   pip install sentence-transformers transformers torch accelerate
# =========================================================
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# ---------------------------------------------------------
# 1. 知識ベース（LLMが絶対に知らない「架空の情報」にするのがコツ）
#    → RAGあり／なしの差がはっきり見える
# ---------------------------------------------------------
documents = [
    "架空のみどり市立あおぞら高校の創立は1987年で、校長は山田花子先生である。",
    "あおぞら高校の文化祭「そら祭」は毎年10月の第3土曜日と日曜日に開催される。",
    "あおぞら高校では、スマートフォンは昼休みと放課後のみ使用してよい。",
    "あおぞら高校の図書室は平日8時から18時まで開いており、1人5冊まで2週間借りられる。",
    "あおぞら高校の制服は紺のブレザーで、夏季は白のポロシャツも認められている。",
    "あおぞら高校の部活動で最も部員が多いのはバドミントン部で、約60人が所属している。",
]

# ---------------------------------------------------------
# 2. 埋め込みモデルで文書をベクトル化（日本語対応モデル）
#    e5系は "query: " / "passage: " の接頭辞を付けるのがルール
# ---------------------------------------------------------
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
doc_vecs = embedder.encode(
    ["passage: " + d for d in documents], normalize_embeddings=True
)

# ---------------------------------------------------------
# 3. 検索（Retrieval）：質問と似ている文書を上位k件取り出す
#    正規化済みなので内積 = コサイン類似度
# ---------------------------------------------------------
def retrieve(query, k=2):
    q_vec = embedder.encode(["query: " + query], normalize_embeddings=True)[0]
    scores = doc_vecs @ q_vec
    top = np.argsort(scores)[::-1][:k]
    return [(documents[i], float(scores[i])) for i in top]

# ---------------------------------------------------------
# 4. 生成（Generation）：小さな日本語対応LLM
#    ※ fine-tuning の授業で作ったモデルに差し替えてもOK
# ---------------------------------------------------------
generator = pipeline(
    "text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto"
)

def ask_llm(prompt):
    messages = [{"role": "user", "content": prompt}]
    out = generator(messages, max_new_tokens=200, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

def answer(query, use_rag=True, k=2):
    if not use_rag:
        return ask_llm(query)

    hits = retrieve(query, k)
    print("【検索された資料】")
    for doc, score in hits:
        print(f"  ({score:.3f}) {doc}")

    context = "\n".join(f"- {doc}" for doc, _ in hits)
    prompt = (
        "以下の資料だけを使って質問に答えてください。"
        "資料に答えがない場合は「資料にはありません」と答えてください。\n\n"
        f"資料:\n{context}\n\n質問: {query}"
    )
    return ask_llm(prompt)

# ---------------------------------------------------------
# 5. RAGあり／なしを比較
# ---------------------------------------------------------
if __name__ == "__main__":
    questions = [
        "あおぞら高校の文化祭はいつですか？",
        "図書室では何冊まで本を借りられますか？",
        "あおぞら高校の校歌の作曲者は誰ですか？",  # 資料にない質問
    ]
    for q in questions:
        print("=" * 60)
        print("質問:", q)
        print("\n▼ RAGなし")
        print(answer(q, use_rag=False))
        print("\n▼ RAGあり")
        print(answer(q, use_rag=True))